In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q openpyxl scikit-learn

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, zipfile
ZIP  = "/content/drive/MyDrive/data/banana_dataset_complete_FINAL (1).zip"
DEST = "/content/banana_dataset"
os.makedirs(DEST, exist_ok=True)
if not os.path.exists(os.path.join(DEST, 'train')):
    with zipfile.ZipFile(ZIP, 'r') as z:
        z.extractall(DEST)
print("Dataset ready:", os.listdir(DEST))

Dataset ready: ['test', 'val', 'train']


In [ ]:
import torch, torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision import transforms as T
from PIL import Image
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Same architecture as ripeness notebook ----
class EfficientNetBranch(nn.Module):
    def __init__(self, in_ch=3, out_dim=512):
        super().__init__()
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        if in_ch != 3:
            old = m.features[0][0]
            m.features[0][0] = nn.Conv2d(in_ch, old.out_channels, old.kernel_size,
                                         old.stride, old.padding, bias=False)
        for p in m.features.parameters(): p.requires_grad = False
        self.backbone = m.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Linear(m.classifier[1].in_features, out_dim),
                                  nn.BatchNorm1d(out_dim), nn.ReLU(), nn.Dropout(0.3))
    def forward(self, x):
        with torch.no_grad(): f = self.pool(self.backbone(x)).flatten(1)
        return self.proj(f)

class MatrixBranch(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64,out_dim), nn.ReLU())
    def forward(self, x): return self.net(x)

class BananaNet(nn.Module):  # ripeness architecture
    def __init__(self):
        super().__init__()
        self.rgb = EfficientNetBranch(3,512)
        self.th  = EfficientNetBranch(3,512)
        self.mat = MatrixBranch(256)
        self.classifier = nn.Sequential(nn.Linear(512+512+256,512), nn.ReLU(),
                                        nn.Dropout(0.4), nn.Linear(512,5))
    def forward(self, rgb, th, mat):
        return self.classifier(torch.cat([self.rgb(rgb), self.th(th), self.mat(mat)], dim=1))

class BananaMultimodalNet(nn.Module):  # adulteration architecture
    def __init__(self, num_classes=2):
        super().__init__()
        self.rgb_branch     = EfficientNetBranch(3,512)
        self.thermal_branch = EfficientNetBranch(3,512)
        self.matrix_branch  = MatrixBranch(256)
        self.fusion     = nn.Sequential(nn.Linear(512+512+256,256), nn.BatchNorm1d(256),
                                        nn.ReLU(), nn.Dropout(0.4))
        self.classifier = nn.Linear(256, num_classes)
    def forward(self, rgb, th, mat):
        f = torch.cat([self.rgb_branch(rgb), self.thermal_branch(th),
                       self.matrix_branch(mat)], dim=1)
        return self.classifier(self.fusion(f))

rip = BananaNet().to(DEVICE)
rip.load_state_dict(torch.load("/content/drive/MyDrive/best_ripeness_model.pth",
                                map_location=DEVICE))
rip.eval()

adult = BananaMultimodalNet(2).to(DEVICE)
ckpt  = torch.load("/content/drive/MyDrive/best_banana_multimodal.pth",
                    map_location=DEVICE, weights_only=False)
adult.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
adult.eval()
print("Both classifiers loaded ✔")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 112MB/s] 


Both classifiers loaded ✔


In [ ]:
import cv2
from tqdm.auto import tqdm

CLASS_NAMES = ['Stage_1_Mature_Green','Stage_2_Green_Yellow_Transition',
               'Stage_3_Early_Ripening','Stage_4_Fully_Yellow','Stage_5_Yellow_Spotted']
CLASS_TO_IDX = {n:i for i,n in enumerate(CLASS_NAMES)}
IMG_SIZE = 224
CACHE_DIR = "/content/matrix_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

eval_tf = T.Compose([T.Resize((256,256)), T.CenterCrop(IMG_SIZE), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

def preprocess_triplet(rgb_path, th_path, mat_path):
    rgb = eval_tf(Image.open(rgb_path).convert('RGB')).unsqueeze(0).to(DEVICE)          # (1,3,224,224)
    th  = eval_tf(Image.open(th_path).convert('L')
                  .resize((IMG_SIZE, IMG_SIZE)).convert('RGB')
                  ).unsqueeze(0).to(DEVICE)                                              # (1,3,224,224)

    # --- matrix ---
    cache = os.path.join(CACHE_DIR, os.path.basename(mat_path).replace('.xlsx','.npy'))
    if os.path.exists(cache):
        mat = np.load(cache)
    else:
        import pandas as pd
        df = pd.read_excel(mat_path, header=None)
        arr = df.map(lambda x: float(str(x).replace('°C','')) if pd.notna(x) else np.nan).values
        mat = np.nan_to_num(arr, nan=0.0)
        mat = cv2.resize(mat, (192, 256))

    mat_t = torch.tensor(mat, dtype=torch.float32)          # (H,W)
    mat_t = mat_t.unsqueeze(0).unsqueeze(0)                  # (1,1,H,W)  ← BATCH dim added
    mat_t = (mat_t - mat_t.mean()) / (mat_t.std() + 1e-6)
    mat_t = torch.nn.functional.interpolate(mat_t, size=(64, 48), mode='bilinear')
    # mat_t is now (1,1,64,48) — KEEP the batch dim
    return rgb, th, mat_t.to(DEVICE)


def probs_for_triplet(rgb, th, mat):
    with torch.no_grad():
        p_rip   = torch.softmax(rip(rgb, th, mat),   dim=1).cpu().numpy()[0]   # (5,)
        p_adult = torch.softmax(adult(rgb, th, mat), dim=1).cpu().numpy()[0]   # (2,)
    return np.concatenate([p_rip, p_adult]).astype(np.float32)                 # (7,)
# Build a flat index of all images (across train/val/test, all stages)
samples = []
for split in ['train','val','test']:
    for stage in CLASS_NAMES:
        sp = os.path.join(DEST, split, stage)
        if not os.path.exists(sp): continue
        for sub, ext in [('RGB','.jpg'), ('Thermal','.JPG'), ('Thermal_Matrix','.xlsx')]:
            pass
        rgb_dir = os.path.join(sp,'RGB'); th_dir = os.path.join(sp,'Thermal'); mat_dir = os.path.join(sp,'Thermal_Matrix')
        rgb_f = sorted(os.listdir(rgb_dir)); th_f = sorted(os.listdir(th_dir)); mat_f = sorted(os.listdir(mat_dir))
        n = min(len(rgb_f), len(th_f), len(mat_f))
        for i in range(n):
            samples.append({
                'stage': CLASS_TO_IDX[stage],
                'rgb': os.path.join(rgb_dir, rgb_f[i]),
                'th':  os.path.join(th_dir,  th_f[i]),
                'mat': os.path.join(mat_dir, mat_f[i]),
            })
print(f"Total samples: {len(samples)}")

# Compute probabilities for every sample (takes ~15-20 min on T4)
probs_cache = []
for s in tqdm(samples, desc="Caching classifier probs"):
    rgb, th, mat = preprocess_triplet(s['rgb'], s['th'], s['mat'])
    probs_cache.append(probs_for_triplet(rgb, th, mat))

np.savez_compressed("/content/drive/MyDrive/lstm_probs_cache.npz",
                    probs=np.stack(probs_cache),
                    stages=np.array([s['stage'] for s in samples]))
print("Cache saved to Drive ✔")

Total samples: 2959


Caching classifier probs:   0%|          | 0/2959 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py

Cache saved to Drive ✔


In [ ]:
z = np.load("/content/drive/MyDrive/lstm_probs_cache.npz")
print("probs shape:", z['probs'].shape)          # MUST be (2959, 7)
print("row-sum check (should be ~2.0):", z['probs'][0].sum())

probs shape: (2959, 7)
row-sum check (should be ~2.0): 2.0


In [ ]:
# Load cached probabilities
z = np.load("/content/drive/MyDrive/lstm_probs_cache.npz")
ALL_PROBS  = z['probs']    # (N, 7)
ALL_STAGES = z['stages']   # (N,)  int 0..4

# Group by stage
by_stage = {i: np.where(ALL_STAGES == i)[0] for i in range(5)}

rng = np.random.default_rng(42)
DAYS_PER_TRANSITION = {0:3, 1:2, 2:2, 3:1}  # stage_i → stage_{i+1}

def make_sequence(start_stage, length):
    """Sample one image per stage in order, starting at start_stage."""
    end_stage = min(start_stage + length - 1, 4)
    seq_stages = list(range(start_stage, end_stage + 1))
    seq_probs, seq_days = [], 0
    for st in seq_stages:
        idx = rng.choice(by_stage[st])
        seq_probs.append(ALL_PROBS[idx])
        seq_days += DAYS_PER_TRANSITION.get(st, 1)
    next_stage = min(end_stage + 1, 4)
    days_to_next = DAYS_PER_TRANSITION.get(end_stage, 1)
    return np.stack(seq_probs), next_stage, days_to_next

SEQS = []
for _ in range(600):
    start  = rng.choice([0, 0, 1, 1, 2])           # mostly early-stage starts
    length = rng.integers(3, 6)                    # 3–5 length
    seq, ns, dn = make_sequence(int(start), int(length))
    SEQS.append({'x': seq, 'next_stage': ns, 'days': dn})

# Train/val split
n_val = 100
rng.shuffle(SEQS)
train_seqs, val_seqs = SEQS[:-n_val], SEQS[-n_val:]
print(f"Train seqs: {len(train_seqs)} | Val seqs: {len(val_seqs)}")

from torch.utils.data import Dataset, DataLoader
class SeqDataset(Dataset):
    def __init__(self, seqs): self.seqs = seqs
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i):
        s = self.seqs[i]
        return (torch.tensor(s['x'],        dtype=torch.float32),
                torch.tensor(s['next_stage'],dtype=torch.long),
                torch.tensor(s['days'],      dtype=torch.float32))

def collate(batch):
    xs   = torch.nn.utils.rnn.pad_sequence([b[0] for b in batch], batch_first=True)
    lens = torch.tensor([len(b[0]) for b in batch])
    ns   = torch.stack([b[1] for b in batch])
    ds   = torch.stack([b[2] for b in batch])
    return xs, lens, ns, ds

dl_train = DataLoader(SeqDataset(train_seqs), batch_size=32, shuffle=True, collate_fn=collate)
dl_val   = DataLoader(SeqDataset(val_seqs),   batch_size=32, shuffle=False, collate_fn=collate)

Train seqs: 500 | Val seqs: 100


In [ ]:
class RipenessForecaster(nn.Module):
    def __init__(self, input_dim=7, hidden=32, layers=2, n_stages=5):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, layers, batch_first=True,
                            dropout=0.2 if layers > 1 else 0.0)
        self.stage_head = nn.Linear(hidden, n_stages)
        self.days_head  = nn.Sequential(nn.Linear(hidden,16), nn.ReLU(), nn.Linear(16,1))
    def forward(self, x, lens):
        packed = torch.nn.utils.rnn.pack_padded_sequence(x, lens, batch_first=True,
                                                          enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = torch.nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        last = out[torch.arange(out.size(0)), lens-1]
        return self.stage_head(last), self.days_head(last).squeeze(-1)

lstm = RipenessForecaster(input_dim=7).to(DEVICE)
opt   = torch.optim.AdamW(lstm.parameters(), lr=1e-3, weight_decay=1e-4)
ce    = nn.CrossEntropyLoss()
mse   = nn.MSELoss()

EPOCHS = 30
best_val = float('inf')
for ep in range(1, EPOCHS+1):
    lstm.train()
    tr_loss = 0
    for xs, lens, ns, ds in dl_train:
        xs, ns, ds = xs.to(DEVICE), ns.to(DEVICE), ds.to(DEVICE)
        opt.zero_grad()
        p_stage, p_days = lstm(xs, lens)
        loss = ce(p_stage, ns) + 0.1 * mse(p_days, ds)
        loss.backward(); opt.step()
        tr_loss += loss.item()
    tr_loss /= len(dl_train)

    lstm.eval()
    vl_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for xs, lens, ns, ds in dl_val:
            xs, ns, ds = xs.to(DEVICE), ns.to(DEVICE), ds.to(DEVICE)
            p_stage, p_days = lstm(xs, lens)
            vl_loss += (ce(p_stage, ns) + 0.1*mse(p_days, ds)).item()
            correct += (p_stage.argmax(1) == ns).sum().item()
            total   += ns.size(0)
    vl_loss /= len(dl_val)
    acc = correct/total
    print(f"Epoch {ep:02d} | train {tr_loss:.4f} | val {vl_loss:.4f} | val acc {acc:.3f}")

    if vl_loss < best_val:
        best_val = vl_loss
        torch.save({'model': lstm.state_dict(), 'input_dim': 7, 'hidden': 32},
                   "/content/drive/MyDrive/lstm_forecaster.pth")
        print("  ↳ saved best")

print("DONE. Checkpoint: /content/drive/MyDrive/lstm_forecaster.pth")

Epoch 01 | train 1.7066 | val 1.6080 | val acc 0.860
  ↳ saved best
Epoch 02 | train 1.4678 | val 1.2425 | val acc 0.860
  ↳ saved best
Epoch 03 | train 0.8924 | val 0.5410 | val acc 0.860
  ↳ saved best
Epoch 04 | train 0.4314 | val 0.4063 | val acc 0.860
  ↳ saved best
Epoch 05 | train 0.3580 | val 0.3650 | val acc 0.860
  ↳ saved best
Epoch 06 | train 0.3290 | val 0.3375 | val acc 0.860
  ↳ saved best
Epoch 07 | train 0.3148 | val 0.3225 | val acc 0.860
  ↳ saved best
Epoch 08 | train 0.2972 | val 0.3075 | val acc 0.860
  ↳ saved best
Epoch 09 | train 0.2777 | val 0.2924 | val acc 0.860
  ↳ saved best
Epoch 10 | train 0.2604 | val 0.2747 | val acc 0.860
  ↳ saved best
Epoch 11 | train 0.2410 | val 0.2534 | val acc 0.860
  ↳ saved best
Epoch 12 | train 0.2165 | val 0.2287 | val acc 0.860
  ↳ saved best
Epoch 13 | train 0.1860 | val 0.2023 | val acc 0.860
  ↳ saved best
Epoch 14 | train 0.1618 | val 0.1792 | val acc 0.930
  ↳ saved best
Epoch 15 | train 0.1362 | val 0.1621 | val acc 0

In [ ]:
import numpy as np

# Save 20 hold-out sequences
demo = val_seqs[:20]

# Find the longest sequence in the demo to use as the padded length
max_len = max(len(d['x']) for d in demo)

# Pad the sequences with zeros to make a uniform numpy array
padded_sequences = []
lengths = []
for d in demo:
    seq = d['x']
    padded = np.zeros((max_len, 7), dtype=np.float32)
    padded[:len(seq)] = seq
    padded_sequences.append(padded)
    lengths.append(len(seq))

# Save as a clean, loadable array
np.savez_compressed("/content/drive/MyDrive/lstm_demo_sequences.npz",
                    sequences=np.array(padded_sequences, dtype=np.float32),
                    lengths=np.array(lengths, dtype=np.int32),
                    next_stage=np.array([d['next_stage'] for d in demo]),
                    days=np.array([d['days'] for d in demo]))

print(f"Demo sequences saved ✔ (Padded to max length: {max_len})")

Demo sequences saved ✔ (Padded to max length: 5)


In [ ]:
def predict_next(sequence_of_probs):
    """
    sequence_of_probs: list/array of shape (T, 7) where each row is
    [p_stage1..p_stage5, p_natural, p_adulterated]
    Returns: {'next_stage': int, 'stage_name': str, 'days_to_next': float}
    """
    seq  = torch.tensor(sequence_of_probs, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    lens = torch.tensor([seq.size(1)])
    lstm.eval()
    with torch.no_grad():
        p_stage, p_days = lstm(seq, lens)
    stage = int(p_stage.argmax(1).item())
    return {'next_stage': stage, 'stage_name': CLASS_NAMES[stage],
            'days_to_next': float(p_days.item())}

# Smoke test
test = predict_next(val_seqs[0]['x'])
print("Predicted:", test)
print("Actual   :", {'next_stage': val_seqs[0]['next_stage'], 'days_to_next': val_seqs[0]['days']})

Predicted: {'next_stage': 4, 'stage_name': 'Stage_5_Yellow_Spotted', 'days_to_next': 0.9949685335159302}
Actual   : {'next_stage': 4, 'days_to_next': 1}
